# 03 · Adquisición y cruce de fuentes
**Proyecto:** Búsqueda de Empleo Sin Experiencia · **Asignatura:** Minería de Datos · **Docente:** Paola Andrea Ortiz  
**Integrantes:** Bryan Samuel Vélez Velásquez · Jhon Jairo Melo Largo

### Pregunta del proyecto (Ficha)
¿Qué características de los jóvenes y de las vacantes aumentan la probabilidad de que una persona **sin experiencia laboral** consiga su primer empleo en Colombia, para decidir hacia qué sectores y formación orientar a estos jóvenes?

**Alcance:** jóvenes de 18 a 28 años en la fuerza de trabajo en Colombia, con datos de 2025 (GEIH) y la demanda laboral registrada en el Servicio Público de Empleo.

### Qué hace este notebook
1. Lee las **2 fuentes** con pandas y verifica su contenido (`.head()`, `.shape`).
2. Prepara la fuente 2 para que tenga una fila por departamento.
3. **Cruza** las fuentes con `pd.merge` usando la columna en común `codigo_departamento` (código DIVIPOLA) y verifica el número de filas antes y después.
4. Construye el **diccionario de datos** del dataset cruzado.
5. Guarda los resultados en `data/processed/`.

### ¿Estás en Google Colab?
Colab no tiene los archivos del proyecto. La siguiente celda **clona el repositorio de GitHub** automáticamente y se ubica en la carpeta `notebooks/`. En VS Code o Jupyter local no hace nada.

In [ ]:
import os, sys
if "google.colab" in sys.modules and not os.path.isdir("../data/raw"):
    if not os.path.isdir("/content/Busqueda-Empleo"):
        !git clone https://github.com/samuelvelez1003/Busqueda-Empleo.git /content/Busqueda-Empleo
    %cd /content/Busqueda-Empleo/notebooks
print("Carpeta de trabajo:", os.getcwd())

## 0. Librerías y rutas

In [ ]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

# Carpeta de datos del repositorio (el notebook se ejecuta desde /notebooks)
for d in ["../data", "data", "../Data", "Data"]:
    if os.path.isdir(d):
        DATA_DIR = d
        break

RUTA_GEIH = os.path.join(DATA_DIR, "processed", "geih2025_jovenes_18_28.csv")
RUTA_SPE  = os.path.join(DATA_DIR, "raw", "spe", "11-Anexo-Demanda-Laboral-Ano-2023.xlsx")
SALIDA    = os.path.join(DATA_DIR, "processed")

print("GEIH:", RUTA_GEIH, "→ existe:", os.path.exists(RUTA_GEIH))
print("SPE :", RUTA_SPE,  "→ existe:", os.path.exists(RUTA_SPE))

## 1. Fuente 1 — GEIH 2025 (DANE)
**Qué es:** Gran Encuesta Integrada de Hogares, la encuesta oficial de mercado laboral de Colombia.  
**De dónde sale:** microdatos del DANE (https://microdatos.dane.gov.co/index.php/catalog/853), enero a diciembre de 2025.  
**Por qué la elegimos:** es la única fuente pública que dice, persona por persona, si un joven está ocupado o desocupado y **si ha trabajado antes**, que es justo nuestro problema.

Los 12 meses se unieron y limpiaron en `02_limpieza.ipynb` (4 módulos por mes → una fila por joven). Aquí leemos ese resultado.

In [ ]:
geih = pd.read_csv(RUTA_GEIH)

print("Forma (filas, columnas):", geih.shape)
geih.head()

In [ ]:
# Verificaciones rápidas
print("Rango de edad:", geih["edad"].min(), "-", geih["edad"].max())
print("Meses:", sorted(geih["mes"].unique()))
print("Departamentos distintos:", geih["codigo_departamento"].nunique())
print()
print(geih["condicion"].value_counts())
print()
print("Jóvenes sin experiencia (aspirantes + recién insertados):", geih["sin_experiencia"].sum())

## 2. Fuente 2 — Vacantes del Servicio Público de Empleo (SPE)
**Qué es:** *Anexo Estadístico de Demanda Laboral*: ofertas de empleo (vacantes) registradas en el SPE por mes, departamento y **experiencia requerida**, y por **nivel educativo requerido**.  
**De dónde sale:** https://www.serviciodeempleo.gov.co/dataempleo-spe/demanda-laboral/anexo-estadistico-de-demanda-laboral-vacantes/  
**Por qué la elegimos:** la GEIH describe a los jóvenes (la oferta de trabajo); el SPE describe **qué piden las empresas** (la demanda). Nos dice cuántas vacantes **no exigen experiencia** en cada departamento.

**Año usado:** 2023 (enero–noviembre), el más reciente que publica el SPE. Es una diferencia de año con la GEIH que documentamos como riesgo.

El archivo Excel tiene varias hojas; usamos **Experiencia** y **Educación**.

In [ ]:
hojas = pd.ExcelFile(RUTA_SPE).sheet_names
print("Hojas del archivo:", hojas)

# Vista cruda de la hoja 'Experiencia' (tal como viene del SPE)
spe_crudo = pd.read_excel(RUTA_SPE, sheet_name="Experiencia", header=None)
print("Forma cruda:", spe_crudo.shape)
spe_crudo.iloc[4:12, :8]

El Excel tiene formato de reporte: una fila con los **años**, otra con los **meses** y columnas para 2015 a 2023. La función `leer_hoja_spe` busca la fila de encabezados, se queda con la columna **Total 2023** y devuelve una tabla ordenada.

In [ ]:
def leer_hoja_spe(hoja, anio=2023):
    """Lee una hoja del anexo del SPE y devuelve: código, departamento, categoría y total de vacantes del año."""
    crudo = pd.read_excel(RUTA_SPE, sheet_name=hoja, header=None)

    # 1) Fila donde empiezan los encabezados ("Código DIVIPOLA")
    fila_enc = crudo.index[crudo[0].astype(str).str.strip() == "Código DIVIPOLA"][0]

    # 2) Fila de años (justo arriba), rellenada hacia la derecha porque las celdas están combinadas
    anios = pd.to_numeric(crudo.loc[fila_enc - 1], errors="coerce").ffill()
    meses = crudo.loc[fila_enc].astype(str).str.strip()

    # 3) Columna 'Total' del año pedido
    col_total = [c for c in crudo.columns if anios[c] == anio and meses[c] == "Total"][0]

    datos = crudo.loc[fila_enc + 1:, [0, 1, 2, col_total]]
    datos.columns = ["codigo", "departamento_spe", "categoria", "vacantes"]
    datos = datos.dropna(subset=["departamento_spe", "categoria"])
    datos["categoria"] = datos["categoria"].astype(str).str.strip()
    datos["vacantes"] = pd.to_numeric(datos["vacantes"], errors="coerce").fillna(0).astype(int)

    # 4) Código DIVIPOLA numérico; 'Total nacional' y 'No especifica' no son departamentos
    datos["codigo_departamento"] = pd.to_numeric(datos["codigo"], errors="coerce")
    datos = datos[~datos["departamento_spe"].str.contains("Total nacional|No especifica", case=False)]
    datos = datos.dropna(subset=["codigo_departamento"])
    datos["codigo_departamento"] = datos["codigo_departamento"].astype(int)
    return datos.drop(columns="codigo")

spe_exp = leer_hoja_spe("Experiencia")
spe_edu = leer_hoja_spe("Educación")

print("Experiencia:", spe_exp.shape, "| Educación:", spe_edu.shape)
spe_exp.head(10)

In [ ]:
print("Categorías de experiencia:", spe_exp["categoria"].unique())
print("Categorías de educación:", spe_edu["categoria"].unique())
print("Departamentos en el SPE:", spe_exp["codigo_departamento"].nunique())

### 2.1 Una fila por departamento
Pasamos de formato largo (departamento × categoría) a formato ancho y calculamos indicadores de la demanda laboral para jóvenes sin experiencia.

In [ ]:
exp = spe_exp.pivot_table(index=["codigo_departamento", "departamento_spe"], columns="categoria",
                          values="vacantes", aggfunc="sum").reset_index()
edu = spe_edu.pivot_table(index="codigo_departamento", columns="categoria",
                          values="vacantes", aggfunc="sum").reset_index()
edu.columns = [str(c).strip() for c in edu.columns]

spe_depto = pd.DataFrame({
    "codigo_departamento": exp["codigo_departamento"],
    "departamento_spe": exp["departamento_spe"],
    "vacantes_total_2023": exp["Total Ofertas de empleo"],
    "vacantes_sin_experiencia_2023": exp["Sin Experiencia"],
    "vacantes_1_6_meses_2023": exp["1 a 6 Meses"],
})
spe_depto["pct_vacantes_sin_experiencia"] = (spe_depto["vacantes_sin_experiencia_2023"] / spe_depto["vacantes_total_2023"] * 100).round(1)
spe_depto["pct_vacantes_hasta_6_meses"] = ((spe_depto["vacantes_sin_experiencia_2023"] + spe_depto["vacantes_1_6_meses_2023"])
                                           / spe_depto["vacantes_total_2023"] * 100).round(1)

spe_depto = spe_depto.merge(edu[["codigo_departamento", "Bachillerato", "Técnica", "Tecnológica", "Universitario", "Total Ofertas de empleo"]],
                            on="codigo_departamento", how="left", validate="1:1")
tot_edu = spe_depto.pop("Total Ofertas de empleo")
spe_depto["pct_vacantes_bachillerato"] = (spe_depto.pop("Bachillerato") / tot_edu * 100).round(1)
spe_depto["pct_vacantes_tecnico_tecnologo"] = ((spe_depto.pop("Técnica") + spe_depto.pop("Tecnológica")) / tot_edu * 100).round(1)
spe_depto["pct_vacantes_universitario"] = (spe_depto.pop("Universitario") / tot_edu * 100).round(1)

print("Forma:", spe_depto.shape)
spe_depto.sort_values("vacantes_total_2023", ascending=False).head(10)

## 3. Cruce de las fuentes
**Columna en común:** `codigo_departamento` (código DIVIPOLA de 2 dígitos del DANE). Está en las dos fuentes porque ambas usan la codificación oficial de departamentos.

**Tipo de cruce:** `left` de GEIH (muchas personas) con SPE (una fila por departamento) → relación **muchos a uno** (`validate="m:1"`). Cada joven recibe los indicadores de demanda laboral de su departamento.

### 3.1 Verificación antes del cruce

In [ ]:
filas_geih_antes = len(geih)
filas_spe = len(spe_depto)

print(f"Filas GEIH antes del cruce: {filas_geih_antes:,}")
print(f"Filas SPE (departamentos): {filas_spe}")
print("¿Código único en el SPE?:", spe_depto["codigo_departamento"].is_unique)

cod_geih = set(geih["codigo_departamento"].unique())
cod_spe = set(spe_depto["codigo_departamento"].unique())
print("Departamentos en GEIH:", len(cod_geih), "| en SPE:", len(cod_spe))
print("En GEIH pero no en SPE:", sorted(cod_geih - cod_spe))
print("En SPE pero no en GEIH:", sorted(cod_spe - cod_geih))

### 3.2 El cruce (`pd.merge`)

In [ ]:
cruzado = pd.merge(
    geih,
    spe_depto,
    on="codigo_departamento",
    how="left",
    validate="m:1",       # falla si el SPE tuviera departamentos repetidos
    indicator=True,       # agrega la columna _merge para verificar
)

print(f"Filas antes: {filas_geih_antes:,} | Filas después: {len(cruzado):,}")
print(cruzado["_merge"].value_counts())

### 3.3 Verificación después del cruce

In [ ]:
assert len(cruzado) == filas_geih_antes, "El cruce cambió el número de filas"
sin_pareja = (cruzado["_merge"] != "both").sum()
print("Filas de la GEIH sin pareja en el SPE:", sin_pareja)

# Los nombres de departamento de ambas fuentes deben coincidir
revision = cruzado[["codigo_departamento", "departamento", "departamento_spe"]].drop_duplicates().sort_values("codigo_departamento")
revision

In [ ]:
cruzado = cruzado.drop(columns=["_merge", "departamento_spe"])
print("Forma final del dataset cruzado:", cruzado.shape)
cruzado.head()

### 3.4 Vista por departamento
Resumen que conecta las dos fuentes: tasa ponderada con la que los jóvenes sin experiencia consiguen empleo (GEIH) frente al porcentaje de vacantes que no exigen experiencia (SPE).

In [ ]:
base = cruzado[cruzado["sin_experiencia"] == 1]

resumen = (base.groupby(["codigo_departamento", "departamento"])
               .apply(lambda g: pd.Series({
                   "jovenes_sin_exp_muestra": len(g),
                   "tasa_empleo_sin_exp_%": np.average(g["consiguio_empleo"], weights=g["factor_expansion"]) * 100,
                   "tasa_empleo_formal_%": np.average((g["empleo_formal"] == "Sí"), weights=g["factor_expansion"]) * 100,
               }), include_groups=False)
               .reset_index()
               .merge(spe_depto.drop(columns="departamento_spe"), on="codigo_departamento", how="left", validate="1:1")
               .round(1)
               .sort_values("pct_vacantes_sin_experiencia", ascending=False))

print("Departamentos en el resumen:", len(resumen))
resumen.head(10)

In [ ]:
corr = resumen[["tasa_empleo_sin_exp_%", "tasa_empleo_formal_%", "pct_vacantes_sin_experiencia",
                "pct_vacantes_tecnico_tecnologo"]].corr().round(2)
corr

## 4. Diccionario de datos del dataset cruzado

In [ ]:
DESCRIPCION = {
    "mes": ("Mes de la encuesta (1 = enero … 12 = diciembre de 2025)", "GEIH", "MES"),
    "id_hogar": ("Identificador anonimizado del hogar (DIRECTORIO-SECUENCIA_P)", "GEIH", "DIRECTORIO, SECUENCIA_P"),
    "orden": ("Número de la persona dentro del hogar", "GEIH", "ORDEN"),
    "factor_expansion": ("Peso de la persona para estimar la población (FEX_C18 / 12)", "GEIH", "FEX_C18"),
    "codigo_departamento": ("Código DIVIPOLA del departamento. LLAVE DEL CRUCE", "GEIH y SPE", "DPTO / Código DIVIPOLA"),
    "departamento": ("Nombre del departamento", "GEIH", "DPTO"),
    "ciudad": ("Ciudad o área metropolitana (Resto del país si no es una de las 32 principales)", "GEIH", "AREA"),
    "zona": ("Urbana (cabecera) o rural y centros poblados", "GEIH", "CLASE"),
    "edad": ("Edad en años cumplidos (18 a 28)", "GEIH", "P6040"),
    "sexo": ("Sexo al nacer", "GEIH", "P3271"),
    "nivel_educativo": ("Mayor nivel educativo alcanzado", "GEIH", "P3042"),
    "nivel_educativo_grupo": ("Nivel educativo agrupado (Básica o menos, Bachiller, Técnico, Tecnólogo, Universitario, Posgrado)", "GEIH", "P3042"),
    "estudia_actualmente": ("¿Asiste actualmente a una institución educativa?", "GEIH", "P6170"),
    "condicion": ("Ocupado o Desocupado", "GEIH", "Módulos Ocupados / No ocupados (DSI)"),
    "ha_trabajado_antes": ("Solo desocupados: ¿ha trabajado alguna vez al menos 2 semanas seguidas?", "GEIH", "P7430"),
    "semanas_buscando": ("Solo desocupados: semanas buscando trabajo", "GEIH", "P7250"),
    "meses_en_empleo_actual": ("Solo ocupados: meses trabajando de forma continua en el empleo actual", "GEIH", "P6426"),
    "posicion_ocupacional": ("Solo ocupados: empleado particular, cuenta propia, doméstico, etc.", "GEIH", "P6430"),
    "tiene_contrato": ("Solo ocupados: ¿tiene contrato (verbal o escrito)?", "GEIH", "P6440"),
    "tipo_contrato": ("Solo ocupados con contrato: término indefinido o fijo", "GEIH", "P6460"),
    "cotiza_pension": ("Solo ocupados: ¿cotiza a un fondo de pensiones?", "GEIH", "P6920"),
    "ingreso_laboral_mensual": ("Solo ocupados: ingreso laboral mensual en pesos", "GEIH", "INGLABO"),
    "sector": ("Solo ocupados: sector económico del empleo (CIIU Rev. 4 agrupado)", "GEIH", "RAMA2D_R4"),
    "empleo_formal": ("Solo ocupados: Sí si cotiza a pensión (aproximación de formalidad)", "GEIH (calculada)", "P6920"),
    "sin_experiencia": ("1 = aspirante (desocupado que nunca ha trabajado) o recién insertado (ocupado ≤ 12 meses en su empleo)", "GEIH (calculada)", "P7430, P6426"),
    "consiguio_empleo": ("VARIABLE OBJETIVO. 1 = recién insertado, 0 = aspirante; vacío si sin_experiencia = 0", "GEIH (calculada)", "P7430, P6426"),
    "vacantes_total_2023": ("Total de vacantes registradas en el SPE en el departamento (ene–nov 2023)", "SPE", "Hoja Experiencia: Total Ofertas de empleo"),
    "vacantes_sin_experiencia_2023": ("Vacantes que no exigen experiencia en el departamento (ene–nov 2023)", "SPE", "Hoja Experiencia: Sin Experiencia"),
    "vacantes_1_6_meses_2023": ("Vacantes que piden de 1 a 6 meses de experiencia (ene–nov 2023)", "SPE", "Hoja Experiencia: 1 a 6 Meses"),
    "pct_vacantes_sin_experiencia": ("% de las vacantes del departamento que no exigen experiencia", "SPE (calculada)", "Hoja Experiencia"),
    "pct_vacantes_hasta_6_meses": ("% de vacantes que piden 6 meses o menos de experiencia", "SPE (calculada)", "Hoja Experiencia"),
    "pct_vacantes_bachillerato": ("% de vacantes que piden bachillerato", "SPE (calculada)", "Hoja Educación"),
    "pct_vacantes_tecnico_tecnologo": ("% de vacantes que piden formación técnica o tecnológica", "SPE (calculada)", "Hoja Educación"),
    "pct_vacantes_universitario": ("% de vacantes que piden formación universitaria", "SPE (calculada)", "Hoja Educación"),
}

diccionario = pd.DataFrame([
    {"columna": col,
     "tipo_dato": str(cruzado[col].dtype),
     "significado": DESCRIPCION[col][0],
     "fuente": DESCRIPCION[col][1],
     "variable_original": DESCRIPCION[col][2],
     "% vacío": round(cruzado[col].isna().mean() * 100, 1),
     "ejemplo": cruzado[col].dropna().iloc[0] if cruzado[col].notna().any() else None}
    for col in cruzado.columns
])

faltan = [c for c in cruzado.columns if c not in DESCRIPCION]
print("Columnas sin descripción:", faltan)
diccionario

## 5. Guardar resultados
- `dataset_cruzado_sin_experiencia.csv`: jóvenes **sin experiencia** (la población de la pregunta) con las variables de ambas fuentes. Es la base para el análisis y el modelo.
- `resumen_departamentos_geih_spe.csv`: una fila por departamento.
- `data/diccionario/diccionario_dataset_cruzado.csv`: el diccionario de arriba.

In [ ]:
final = cruzado[cruzado["sin_experiencia"] == 1]
final.to_csv(os.path.join(SALIDA, "dataset_cruzado_sin_experiencia.csv"), index=False, encoding="utf-8-sig")
resumen.to_csv(os.path.join(SALIDA, "resumen_departamentos_geih_spe.csv"), index=False, encoding="utf-8-sig")
diccionario.to_csv(os.path.join(DATA_DIR, "diccionario", "diccionario_dataset_cruzado.csv"), index=False, encoding="utf-8-sig")

print("Dataset cruzado (sin experiencia):", final.shape)
print("Resumen por departamento:", resumen.shape)

## 6. Riesgos y dificultades encontradas
| Riesgo / dificultad | Cómo lo resolvimos o lo vamos a resolver |
|---|---|
| **La GEIH no pregunta “¿es su primer empleo?”** | Aproximación: aspirantes (nunca han trabajado) vs. recién insertados (≤ 12 meses en el empleo). Documentada en `02_limpieza`. |
| **Diferencia de años**: GEIH 2025 vs. SPE 2023 (el SPE no ha publicado anexos más recientes) | Usamos el SPE como indicador de la **estructura** de la demanda por departamento (qué tan común es pedir cero experiencia), que cambia lento. Lo actualizaremos si el SPE publica 2024–2025. |
| **Cruce solo por departamento** | Las fuentes no comparten un identificador de persona. El cruce es de contexto: cada joven recibe los indicadores de su departamento. |
| **El SPE solo registra vacantes publicadas en el SPE** | Sesgo: faltan ofertas de portales privados e informales. Lo tenemos en cuenta al interpretar. |
| **Desbalance de la variable objetivo** (muchos más recién insertados que aspirantes) | Lo trataremos en el modelo (pesos por clase, métricas como AUC en vez de exactitud). |
| **Formatos del DANE** (archivos con `;`, `latin1`, nombres con caracteres especiales) | Lectura con `sep=";"`, `encoding="latin1"` y nombres de archivo normalizados. |

## 7. Conclusiones
*(Completar después de ejecutar.)*
1. Las dos fuentes se leyeron y se cruzaron sin perder filas: …
2. Departamentos con más vacantes sin experiencia: …
3. ¿Hay relación entre las vacantes sin experiencia y la tasa de empleo de los jóvenes? …

**Siguiente paso:** `04_analisis_descriptivo.ipynb`.